In [1]:
import sys
print(sys.executable)

/Users/masood/Desktop/expedia-ranking/.venv/bin/python


In [2]:
import duckdb

con = duckdb.connect()   # in-memory DuckDB database

df_peek = con.sql("""
    SELECT *
    FROM 'data/train.csv'
    LIMIT 20
""").df()

df_peek

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,comp6_rate_percent_diff,comp7_rate,comp7_inv,comp7_rate_percent_diff,comp8_rate,comp8_inv,comp8_rate_percent_diff,click_bool,gross_bookings_usd,booking_bool
0,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,893,3,3.5,...,NULL,NULL,NULL,NULL,0,0,NULL,0,NULL,0
1,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,10404,4,4.0,...,NULL,NULL,NULL,NULL,0,0,NULL,0,NULL,0
2,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,21315,3,4.5,...,NULL,NULL,NULL,NULL,0,0,NULL,0,NULL,0
3,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,27348,2,4.0,...,NULL,NULL,NULL,NULL,-1,0,5,0,NULL,0
4,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,29604,4,3.5,...,NULL,NULL,NULL,NULL,0,0,NULL,0,NULL,0
5,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,30184,4,4.5,...,NULL,NULL,NULL,NULL,0,0,7,0,NULL,0
6,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,44147,3,3.5,...,NULL,NULL,NULL,NULL,0,0,NULL,0,NULL,0
7,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,50984,2,0.0,...,NULL,NULL,NULL,NULL,NULL,NULL,NULL,0,NULL,0
8,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,53341,4,4.0,...,NULL,NULL,NULL,NULL,0,0,6,0,NULL,0
9,1,2013-04-04 08:32:15,12,187,NULL,NULL,219,56880,4,4.0,...,NULL,NULL,NULL,NULL,0,0,NULL,0,NULL,0


In [3]:
con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT srch_id) AS n_searches
    FROM 'data/train.csv'
""").df()

,n_rows,n_searches
0,9917530,399344


In [4]:
con.sql("""
    CREATE OR REPLACE TABLE sample AS
    WITH searches AS (
        SELECT DISTINCT srch_id
        FROM 'data/train.csv'
    ),
    sampled AS (
        SELECT srch_id
        FROM searches
        USING SAMPLE reservoir(50000 ROWS) REPEATABLE (42)
    )
    SELECT t.*
    FROM 'data/train.csv' AS t
    JOIN sampled USING (srch_id)
""")

con.sql("COPY sample TO 'data/sample_50k.parquet' (FORMAT parquet)")

con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT srch_id) AS n_searches
    FROM sample
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_searches
0,1240472,50000
